# 🔄 Module 5 (Advanced): Accumulating Snapshot Fact Table

## Overview

In this notebook we build an **accumulating snapshot** fact table — the third and most sophisticated fact table pattern.

**What you'll learn:**
- Understand the accumulating snapshot pattern and when to use it
- Model a process pipeline with multiple milestones
- Build `fact_trip_lifecycle` tracking each trip through its stages
- Query for pipeline analytics: duration buckets, in-flight trips, milestone lag

---

**Prerequisites:**
- Completed Module 3 (Gold Layer)
- `silver_trips`, `dim_date`, `dim_time`, and `dim_station` tables exist


## 🎓 Concept: Accumulating Snapshot

An accumulating snapshot models a **business process** with a defined lifecycle.

| Pattern | Row behaviour | Best for |
|---------|--------------|----------|
| Transaction fact | One row per event, append-only | Raw event log |
| Periodic snapshot | New rows each period, immutable | Status at a point in time |
| **Accumulating snapshot** | One row per instance, **updated** as milestones are reached | Pipeline, order fulfilment, trip lifecycle |

### Oslo Bysykkel Lifecycle

For a bike trip the milestones are:

```
Milestone 1          Milestone 2          Milestone 3
Bike unlocked   →    Bike returned   →    Record processed
(started_at)         (ended_at)           (ingestion time)
```

In a production system you would `MERGE` new data into this table, updating existing rows.  
In this course we simulate that by populating all milestone columns from the already-complete `silver_trips`.


## Step 1: Verify Source Tables

In [ ]:
%%sql
SELECT
    COUNT(*)                    AS total_trips,
    COUNT(CASE WHEN ended_at IS NOT NULL THEN 1 END)    AS complete_trips,
    COUNT(CASE WHEN ended_at IS NULL     THEN 1 END)    AS in_flight_trips
FROM silver_trips
WHERE is_valid = TRUE

## Step 2: Create `fact_trip_lifecycle` (Accumulating Snapshot)

In [ ]:
%%sql
-- ============================================================
-- FACT TABLE: fact_trip_lifecycle  (Accumulating Snapshot)
-- Grain : one row per individual trip
-- ============================================================
CREATE OR REPLACE TABLE fact_trip_lifecycle
USING DELTA
AS
SELECT
    -- Surrogate trip key (row number as stable identifier)
    MONOTONICALLY_INCREASING_ID()                                       AS trip_key,

    -- ── Milestone 1: Trip started ─────────────────────────────────────
    CAST(DATE_FORMAT(started_at, 'yyyyMMdd') AS INT)                    AS start_date_key,
    CAST(DATE_FORMAT(started_at, 'HHmm')    AS INT)                    AS start_time_key,
    started_at                                                          AS milestone_1_started_at,

    -- ── Milestone 2: Trip ended ───────────────────────────────────────
    CAST(DATE_FORMAT(ended_at,   'yyyyMMdd') AS INT)                    AS end_date_key,
    CAST(DATE_FORMAT(ended_at,   'HHmm')    AS INT)                    AS end_time_key,
    ended_at                                                            AS milestone_2_ended_at,

    -- ── Milestone 3: Record ingested/processed ────────────────────────
    -- In production this would be filled later by the ETL pipeline.
    -- We simulate it as silver processing time (current_timestamp of load).
    CURRENT_TIMESTAMP()                                                 AS milestone_3_processed_at,

    -- ── Station foreign keys ──────────────────────────────────────────
    ds_start.station_key                                                AS start_station_key,
    ds_end.station_key                                                  AS end_station_key,

    -- ── Milestone lag measures (how long between milestones) ──────────
    duration                                                            AS m1_to_m2_seconds,   -- trip duration
    ROUND(duration / 60.0, 2)                                           AS m1_to_m2_minutes,

    -- Lag from end of trip to processing (simulated in this exercise)
    UNIX_TIMESTAMP(CURRENT_TIMESTAMP()) - UNIX_TIMESTAMP(ended_at)     AS m2_to_m3_seconds,

    -- ── Derived attributes ────────────────────────────────────────────
    CASE
        WHEN duration < 600    THEN 'Short   (<10 min)'
        WHEN duration < 1800   THEN 'Medium  (10-30 min)'
        WHEN duration < 3600   THEN 'Long    (30-60 min)'
        ELSE                        'Very Long (>60 min)'
    END                                                                 AS duration_bucket,

    -- Did the trip cross midnight?
    CASE WHEN DATE(started_at) != DATE(ended_at) THEN TRUE ELSE FALSE END AS is_overnight_trip,

    -- Are all milestones present? (in production some rows may still be in-flight)
    CASE
        WHEN started_at IS NOT NULL
         AND ended_at   IS NOT NULL THEN TRUE
        ELSE FALSE
    END                                                                 AS is_complete

FROM silver_trips st
JOIN dim_station ds_start ON st.start_station_id = ds_start.station_id
JOIN dim_station ds_end   ON st.end_station_id   = ds_end.station_id
WHERE st.is_valid = TRUE

## Step 3: Verify the Table

In [ ]:
%%sql
SELECT
    COUNT(*)                                            AS total_rows,
    SUM(CASE WHEN is_complete    THEN 1 ELSE 0 END)    AS complete_trips,
    SUM(CASE WHEN is_overnight_trip THEN 1 ELSE 0 END) AS overnight_trips,
    ROUND(AVG(m1_to_m2_minutes), 1)                    AS avg_duration_minutes
FROM fact_trip_lifecycle

## Step 4: Analytical Queries

In [ ]:
%%sql
-- Trip volume by duration bucket
SELECT
    duration_bucket,
    COUNT(*)                                AS trip_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
FROM fact_trip_lifecycle
GROUP BY duration_bucket
ORDER BY MIN(m1_to_m2_seconds)

In [ ]:
%%sql
-- Milestone lag analysis: how quickly were trips processed?
SELECT
    PERCENTILE_APPROX(m2_to_m3_seconds / 3600.0, 0.5)  AS median_hours_to_process,
    PERCENTILE_APPROX(m2_to_m3_seconds / 3600.0, 0.95) AS p95_hours_to_process,
    MAX(m2_to_m3_seconds / 3600.0)                      AS max_hours_to_process
FROM fact_trip_lifecycle
WHERE is_complete = TRUE

In [ ]:
%%sql
-- Top 10 station-to-station routes by trip count
SELECT
    s.station_name  AS start_station,
    e.station_name  AS end_station,
    COUNT(*)        AS trips,
    ROUND(AVG(f.m1_to_m2_minutes), 1) AS avg_minutes
FROM fact_trip_lifecycle f
JOIN dim_station s ON f.start_station_key = s.station_key
JOIN dim_station e ON f.end_station_key   = e.station_key
GROUP BY s.station_name, e.station_name
ORDER BY trips DESC
LIMIT 10

In [ ]:
%%sql
-- How does trip duration vary by time of day?
SELECT
    t.time_period,
    COUNT(*)                                AS trips,
    ROUND(AVG(f.m1_to_m2_minutes), 1)      AS avg_duration_min
FROM fact_trip_lifecycle f
JOIN dim_time t ON f.start_time_key = t.time_key
GROUP BY t.time_period
ORDER BY avg_duration_min DESC

## 📌 Key Takeaways

- An **accumulating snapshot** has **one row per process instance** that gets updated as milestones are reached
- The table contains **multiple date/time foreign keys** — one per milestone
- **Lag columns** between milestones expose process efficiency (e.g. processing delays)
- In production, you'd use `MERGE INTO` to update rows when new milestone data arrives
- `is_complete = FALSE` identifies "in-flight" records still awaiting later milestones
